In [ ]:

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, ttest_ind

# Load data

In [9]:
input_dir = "/mnt/c/users/helen/Desktop/FIBERS"

In [ ]:
dfs = []

for root, dirs, files in os.walk(input_dir):
    for filename in files:
        if filename.lower().endswith(".csv"):

            path = os.path.join(root, filename)

            df = pd.read_csv(path)
            
            # If everything ended up in one column, try semicolon
            if df.shape[1] == 1:
                df = pd.read_csv(path, sep=";")

            # Optional metadata
            df["File"] = os.path.splitext(filename)[0].replace(" ", "_")
            df["Path"] = path

            dfs.append(df) 

# Combine all tables
data = pd.concat(dfs, ignore_index=True)

# Create ROI column
data['ROI'] = data['Label'].apply(lambda x: x.split(":")[1])
data['Sample_name'] = data['File'].apply(lambda x: x.split("_")[1].split("-")[0])

# Delete first 3 columns
data.drop(data.columns[[0, 1, 2]], axis=1, inplace=True)

# Reorder columns
data = data[
    [
        "Sample_name",
        "File",
        "Measurement_type",
        "Length",
        "ROI",
        "Path"
    ]
]

AttributeError: 'float' object has no attribute 'split'

In [11]:
data

,,Label,Angle,Length,Measurement_type,File,Path,;Label;Angle;Length;Measurement_type
0,1.0,siORC1_MGS1-02_merged.png:0303-0404,-30.579,51.108,Fiber_length,siORC1_MGS1-02_Fiber_length,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...,NaN
1,2.0,siORC1_MGS1-02_merged.png:0273-0360,-36.254,37.202,Fiber_length,siORC1_MGS1-02_Fiber_length,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...,NaN
2,NaN,NaN,NaN,NaN,NaN,siORC1_MGS1-03_Fiber_length,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...,1;siORC1_MGS1-03_merged.png:0944-0253;-19.093;...
3,NaN,NaN,NaN,NaN,NaN,siORC1_MGS1-03_Fiber_length,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...,2;siORC1_MGS1-03_merged.png:0968-0321;-21.490;...
4,NaN,NaN,NaN,NaN,NaN,siORC1_MGS1-03_Fiber_length,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...,3;siORC1_MGS1-03_merged.png:0581-0161;-36.870;...
...,...,...,...,...,...,...,...,...
967,3.0,HCl_1h15min_15o_2_-11_merged.png:0124-0241,-21.580,96.067,Interorigin_distance,HCl_1h15min_15o_2_-11_Interorigin_distance,/mnt/c/users/helen/Desktop/FIBERS/WT_fibers_sp...,NaN
968,1.0,HCl_1h15min_15o_2_-12_merged.png:0477-0459,7.838,124.665,Interorigin_distance,HCl_1h15min_15o_2_-12_Interorigin_distance,/mnt/c/users/helen/Desktop/FIBERS/WT_fibers_sp...,NaN
969,1.0,HCl_1h15min_15o_2_-16_merged.png:0144-0607,-20.241,170.532,Interorigin_distance,HCl_1h15min_15o_2_-16_Interorigin_distance,/mnt/c/users/helen/Desktop/FIBERS/WT_fibers_sp...,NaN
970,1.0,HCl_1h15min_15o_2_-17_merged.png:0126-0721,58.201,147.076,Interorigin_distance,HCl_1h15min_15o_2_-17_Interorigin_distance,/mnt/c/users/helen/Desktop/FIBERS/WT_fibers_sp...,NaN


In [ ]:
# Fix Sample name manually for some dataframe
data["Sample_name"] = data["Sample_name"].apply(
    lambda x: "WT" if "1h15min" in x else x
)

In [ ]:
# Split data into 2 dataframes
speed = data[data['Measurement_type']=='Fiber_length']
iod = data[data['Measurement_type']=='Interorigin_distance']

In [ ]:
iod.head()

In [ ]:
speed.head()

# Process replication speed

In [ ]:
conversion_factor = 2.59 # kb/µm
time = 20 # minutes

In [ ]:
# Checking speed file
counts = speed.groupby("File").size()
odd_files = counts[counts % 2 != 0].index.tolist()

if len(odd_files) == 0:
    print("All files contain an even number of fibers.")
else:
    print("The following files contain an odd number of fibers will be removed:")
    print(*odd_files, sep="\n")
    
    # Removing odd files from speed dataframe
    speed = speed[~speed["File"].isin(odd_files)].copy()

In [ ]:
# Add extra inedex to group pairs of files
speed["Index"] = speed.groupby("File").cumcount() // 2

# Calculate sum of fiber length in pairs
speed_processed = speed.groupby(["File", "Index"], as_index=False).agg(
        Total_Length=("Length", "sum"),
        ROI=("ROI", list),
        Path=("Path", "first"),
        Sample_name=("Sample_name", "first")
        )

# Convert speed to kb/min
speed_processed['Speed_kb_min'] = speed_processed['Total_Length'].apply(lambda x: x * conversion_factor / time)

# Delete extra columns
replication_speed = speed_processed[['Sample_name', 'File', 'Speed_kb_min', 'ROI', 'Path']]

# Info
#n_fibers = len(replication_speed)
#print(f"The amount of fibers is: {n_fibers}")

# Sample names
speed_sample_names = set(replication_speed['Sample_name'])
print(f"There are the following samples: {speed_sample_names}")

In [ ]:
replication_speed.head()

# Process IOD

In [ ]:
iod['IOD_kb'] = iod['Length'].apply(lambda x: x * conversion_factor)
iod_kb = iod[["Sample_name", "File", 'IOD_kb', 'ROI', 'Path']]

# Info
#n_origins = len(iod_kb)
#print(f"The amount of origins is: {n_origins}")

# Samples names
iod_sample_names = set(iod_kb['Sample_name'])
print(f"There are the following samples: {iod_sample_names}")

# Graphs

## Replication speed graph

In [ ]:
plt.figure(figsize=(7, 5))

# Variables
data_plot = replication_speed
var = "Speed_kb_min"

# Order of groups (optional)
sample_order = ["WT", "MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]

groups = []
labels = []

for sample in sample_order:
        values = data_plot.loc[
        data_plot["Sample_name"] == sample,
        var
    ]
        
        groups.append(values)
        labels.append(sample)

    

bp = plt.boxplot(
    groups,
    patch_artist=True,
    showfliers=False,
    widths=0.6,
)

for box in bp["boxes"]:
    box.set(facecolor="#4C72B0", alpha=0.6)

# Jittered dots
for i, values in enumerate(groups, start=1):
    x = np.random.normal(i, 0.05, len(values))
    plt.scatter(x, values, s=20, color="black", alpha=0.7, zorder=3)

plt.ylabel("Replication speed (kb/min)")
plt.xticks(range(1, len(labels) + 1), labels)
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(f"{input_dir}/replication_speed_boxplot.png", dpi=600, bbox_inches="tight")
print(f"Plot is saved in the directory: {input_dir}")

plt.show()

## IOD graph

In [ ]:
plt.figure(figsize=(7, 5))

# Variables
data_plot = iod_kb
var = "IOD_kb"

# Order of groups (optional)
sample_order = ["WT", "MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]

groups = []
labels = []

for sample in sample_order:
        values = data_plot.loc[
        data_plot["Sample_name"] == sample,
        var
    ]
        
        groups.append(values)
        labels.append(sample)

    

bp = plt.boxplot(
    groups,
    patch_artist=True,
    showfliers=False,
    widths=0.6,
)

for box in bp["boxes"]:
    box.set(facecolor="#4C72B0", alpha=0.6)

# Jittered dots
for i, values in enumerate(groups, start=1):
    x = np.random.normal(i, 0.05, len(values))
    plt.scatter(x, values, s=20, color="black", alpha=0.7, zorder=3)

plt.ylabel("iod_kb")
plt.xticks(range(1, len(labels) + 1), labels)
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(f"{input_dir}/replication_speed_boxplot.png", dpi=600, bbox_inches="tight")
print(f"Plot is saved in the directory: {input_dir}")

plt.show()

# Check outliers

## IOD

In [ ]:
Q1 = iod_kb['IOD_kb'].quantile(0.25)
Q3 = iod_kb['IOD_kb'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = iod_kb[(iod_kb['IOD_kb'] < lower) | (iod_kb['IOD_kb'] > upper)]

In [ ]:
outliers

# Statistical analysis

## U-test

In [ ]:
# Data
data_plot = iod_kb
var = "IOD_kb"

wt = data_plot.loc[
    data_plot["Sample_name"] == "WT",
    var
].dropna()

results = []

for sample in ["MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]:

    mutant = data_plot.loc[
        data_plot["Sample_name"] == sample,
        var
    ].dropna()

    stat, p = mannwhitneyu(
        wt,
        mutant
    )

    results.append({
        "Comparison": f"WT vs {sample}",
        "WT_n": len(wt),
        "Sample_n": len(mutant),
        "U": stat,
        "p-value": p,
    })

stats_u_df = pd.DataFrame(results)

print(stats_u_df)

## T-Test

In [ ]:
# Data
data_plot = iod_kb
var = "IOD_kb"

wt = data_plot.loc[
    data_plot["Sample_name"] == "WT",
    var
].dropna()

results = []

for sample in ["MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]:

    mutant = data_plot.loc[
        data_plot["Sample_name"] == sample,
        var
    ].dropna()

    stat, p = ttest_ind(
        wt,
        mutant,
        equal_var=False,   # Welch's t-test (recommended)
    )

    results.append({
        "Comparison": f"WT vs {sample}",
        "WT_n": len(wt),
        "Sample_n": len(mutant),
        "U": stat,
        "p-value": p,
    })

stats_t_df = pd.DataFrame(results)

print(stats_t_df)

# Statistical tables export

In [ ]:

replication_speed.to_excel(f"{input_dir}/replication_speed.xlsx", index=False)
iod_kb.to_excel(f"{input_dir}/iod_kb.xlsx", index=False)